# K4: vektori reči + logistička regresija + SMOTE (TripAdvisor hoteli)
**Tip:** Blok 3, zadatak za vežbu. Reprezentacija: vektori reči (`en_core_web_md`) kroz sopstvenu klasu transformera. Klasifikator: logistička regresija. Balansiranje: SMOTE.

### Zadatak
U fajlu `tripadvisor_hotel_reviews.csv` su komentari gostiju hotela sa TripAdvisor-a (kolone `Review` i `Rating`). Izvor: Kaggle "TripAdvisor Hotel Reviews".
- Od kolone `Rating` napraviti `sentiment` (1 i 2 negativan, 3 neutralan, 4 i 5 pozitivan).
- Transformisati `Review` primenom **vektora reči iz spaCy medium modela**. Transformaciju napisati kao klasu koja nasleđuje `BaseEstimator` i `TransformerMixin`.
- Prvi model: **logistička regresija** sa podrazumevanim hiperparametrima, uz **SMOTE**.
- Metrike po klasama i ukupno, interpretacija.
- Drugi model: komentar sa opsezima, pa podešavanje dva hiperparametra (transformacija nema hiperparametre, pa npr. `C` i `k_neighbors` kod SMOTE-a) kroz 5-fold kros-validaciju.
- Metrike za podešeni model i poređenje.

In [ ]:
import numpy as np
import pandas as pd

from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer

from sklearn.model_selection import train_test_split, GridSearchCV, RandomizedSearchCV
from sklearn.preprocessing import RobustScaler, OneHotEncoder, MinMaxScaler, FunctionTransformer
from sklearn import metrics
from sklearn.metrics import classification_report, ConfusionMatrixDisplay, balanced_accuracy_score
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline   # imblearn Pipeline radi i bez SMOTE-a, pa moze da ostane uvek

from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.dummy import DummyClassifier

import matplotlib.pyplot as plt
import seaborn as sb

from pathlib import Path
from IPython.display import display

import spacy

RAND_STATE = 1
DATA_DIR = Path.cwd().parent / 'data'

### Šta je urađeno
- Učitani podaci, izbačeni prazni i duplikati.
- Uzet uzorak od 4000 komentara (`random_state=1`) zbog brzine računanja vektora.
- Napravljena izlazna varijabla `sentiment` (pozitivnih oko 73%, negativnih 16%, neutralnih 11%).
- Dodata i pregledana `review_len` (kdeplot). Odluka: ulaz je samo `Review` (dataset nema drugih korisnih kolona).
- Napomena: tekst je već sveden na mala slova i bez većine interpunkcije, ali spaCy čišćenje (lema, stop reči) i dalje radiš.
- Podela 80:20 sa `stratify`.

In [ ]:
data = pd.read_csv(DATA_DIR / 'tripadvisor_hotel_reviews.csv')
data.info()

In [ ]:
data = data.dropna(subset=['Review']).drop_duplicates(subset='Review')
data = data.sample(n=4000, random_state=RAND_STATE).reset_index(drop=True)
print(data['Rating'].value_counts().sort_index())

In [ ]:
def rating_to_sentiment(rating: int):
    if rating < 3:
        return 'negativan'
    return 'neutralan' if rating == 3 else 'pozitivan'

data['sentiment'] = data['Rating'].apply(rating_to_sentiment)
print(data['sentiment'].value_counts(normalize=True))

In [ ]:
data['review_len'] = data['Review'].str.split().str.len()
print(data['review_len'].describe())

plt.figure(figsize=(10, 6))
sb.kdeplot(data=data, x='review_len', hue='sentiment',
           fill=True, common_norm=False, alpha=0.5, linewidth=2)
plt.title('Duzina komentara po sentimentu')
plt.grid(True, linestyle='--', alpha=0.7)
plt.show()

In [ ]:
X = data['Review']
y = data['sentiment']
y_encodings, y_levels = pd.factorize(y)
print(list(y_levels))

X_train, X_test, y_train, y_test = train_test_split(X, y_encodings,
                                                    test_size=0.2,
                                                    random_state=RAND_STATE,
                                                    stratify=y_encodings)
print(X_train.shape, X_test.shape)

---
# ▶ OVDE KREĆEŠ

Sve iznad je gotovo. Od ove ćelije pa nadalje kucaš sam: kod, komentare u markdown ćelijama i tumačenja.

**Tvoji koraci:**

1. `spacy.load('en_core_web_md', disable=['parser','ner'])` i funkcija čišćenja
2. Klasa `ReviewVectorizer(BaseEstimator, TransformerMixin)` sa `__init__`, `fit` (vraća `self`) i `transform` (matrica n x 300)
3. `SMOTE` + `LogisticRegression(max_iter=2000)` u imblearn `Pipeline`, pa `fit`
4. Metrike i tumačenje (posebno neutralna klasa)
5. Komentar o hiperparametrima, `GridSearchCV` npr. `classifier__C` + `resampling__k_neighbors`
6. Podešeni model i poređenje (opciono dodaj `DummyClassifier` kao donju granicu)

---
## Dati kod
Ovo dobijaš uz zadatak. Pokreni ćeliju kad stigneš do poređenja modela.

In [ ]:
# DATI KOD (dobija se na kolokvijumu): poredjenje modela
def get_model_metrics(y_true: np.ndarray, y_pred: np.ndarray, model_name: str):
    report = classification_report(y_true, y_pred, output_dict=True)
    macro_avg = report['macro avg']
    weighted_avg = report['weighted avg']
    return {
        'Model': model_name,
        'Macro Precision': macro_avg['precision'],
        'Macro Recall': macro_avg['recall'],
        'Macro F1-Score': macro_avg['f1-score'],
        'Weighted Precision': weighted_avg['precision'],
        'Weighted Recall': weighted_avg['recall'],
        'Weighted F1-Score': weighted_avg['f1-score'],
        'Balanced Accuracy': balanced_accuracy_score(y_true, y_pred)
    }